# TradePose Market Profile Strategy Guide

這份 notebook 是給新使用者或 LLM 使用的端到端範例：先用 `BatchTester` 查詢 instrument，再建立一個極簡 `StrategyConfig`，透過 OHLCV 下載 Market Profile 指標，最後簡短帶到 trades 產生流程。

使用順序：
1. 安裝套件與設定 API key。
2. 查詢 XAUUSD instrument。
3. 建立極簡策略：`open > VAH` 進場、`open < POC` 出場。
4. 下載含 Market Profile 指標的 OHLCV。
5. 查看 Market Profile struct 與 `tpo_distribution` 內部值。
6. 用 `submit()` 產生 trades，後續可分析 MAE / MFE / PnL。


In [ ]:
# Colab / Jupyter dependency setup
# 本機 VS Code 使用 uv sync 後的 .venv kernel 時可略過。
!pip install tradepose-client --ignore-requires-python --quiet


In [5]:
import os
from getpass import getpass

import polars as pl
from dotenv import load_dotenv

from tradepose_client import BatchTester, Freq, TradeDirection
from tradepose_client.batch import Period

# 本機使用 repo root 的 .env；Colab 也可以上傳或手動建立 .env。
load_dotenv()

API_KEY = os.getenv("TRADEPOSE_API_KEY") or getpass("TradePose API key: ")
SERVER_URL = os.getenv("TRADEPOSE_SERVER_URL", "https://api.tradepose.com")

tester = BatchTester(api_key=API_KEY, server_url=SERVER_URL, poll_interval=2.0)
print(f"Connected to {SERVER_URL}")


2026-04-27 22:53:50,771 - tradepose_client.utils - INFO - 📓 Jupyter environment detected - nest_asyncio automatically applied for seamless async support


Connected to https://api.tradepose.com


In [6]:
# 下載/查詢 instrument metadata。
# 先用 symbol 模糊搜尋，確認實際可用的 instrument key 與 tick_size。
instruments = tester.list_instruments(symbol="XAUUSD", limit=20)

print(f"count={instruments.count}, total={instruments.total}")
for idx, inst in enumerate(instruments.instruments):
    print(idx, inst.key, inst)


2026-04-27 22:53:53,331 - tradepose_client.resources.instruments - INFO - Listed 4 instruments (total: 4)


count=4, total=4
0 FTMO:swap:XAUUSD id=59 symbol='XAUUSD' account_source='FTMO' broker_type='mt5' market_type='swap' base_currency='USD' quote_currency='USD' tick_size='0.0100000000' lot_size='0.0100000000' price_precision=2 quantity_precision=2 contract_size='1.0000000000' point_value='100.00000000' status='active' created_at=datetime.datetime(2026, 1, 30, 7, 40, 20, 721108, tzinfo=TzInfo(0)) updated_at=datetime.datetime(2026, 4, 27, 10, 15, 18, 390437, tzinfo=TzInfo(0))
1 PEPPERSTONE:spot:XAUUSD id=264845 symbol='XAUUSD' account_source='PEPPERSTONE' broker_type='mt5' market_type='spot' base_currency='XAU' quote_currency='USD' tick_size='0.0100000000' lot_size='0.0100000000' price_precision=2 quantity_precision=2 contract_size='1.0000000000' point_value='100.00000000' status='active' created_at=datetime.datetime(2026, 3, 18, 7, 48, 35, 280737, tzinfo=TzInfo(0)) updated_at=datetime.datetime(2026, 4, 27, 10, 15, 11, 618030, tzinfo=TzInfo(0))
2 PEPPERSTONE:spot:XAUUSD-F id=266461 symbol=

## Strategy Code

這個策略只保留觀察指標需要的最小結構：

- Daily Market Profile
- Initial Balance Market Profile
- ATR
- `open > VAH` 進場、`open < POC` 出場

`volatility_indicator` 會用 ATR，後續 trades 分析可用它正規化 MAE / MFE，例如 `mae / ATR`。`volatility_level` 是市場 regime 分層，不一定要設定；本範例保留一個簡單 ATR quantile 版本，設定後 trades 會帶出相關 level 欄位，方便依低/中/高波動分組。


In [7]:
"""Simple Market Profile Strategy

目標是展示如何下載 OHLCV 與 Market Profile 指標，而不是展示複雜交易邏輯。
進出場規則刻意簡化為：open > VAH 進場，open < POC 出場。
"""

from zoneinfo import ZoneInfo

import polars as pl
from pydantic import Field
from tradepose_client import BlueprintBuilder, Freq, IndicatorType, StrategyBuilder, TradeDirection, TrendType
from tradepose_models.indicators.market_profile import (
    create_daily_mode,
    create_intraday_mode,
    create_profile_shape_config,
)
from tradepose_models.strategy import StrategyConfig, StrategyParams, VolatilityLevelConfig


class SimpleMarketProfileParams(StrategyParams):
    """極簡 Market Profile 範例策略參數。"""

    instrument: str = Field(description="商品代碼，如 PEPPERSTONE:spot:XAUUSD")
    base_freq: Freq = Field(description="基準頻率")
    trade_direction: TradeDirection = Field(description="交易方向")
    tz: ZoneInfo = Field(default_factory=lambda: ZoneInfo("UTC"), description="策略時區")
    ib_start_hour: int = Field(default=9, ge=0, le=23, description="IB 起始小時")
    ib_end_hour: int = Field(default=11, ge=0, le=23, description="IB 結束小時")
    tick_size: float = Field(default=1.0, gt=0, description="Market Profile tick size")
    atr_freq: Freq = Field(default=Freq.HOUR_1, description="ATR 頻率")
    atr_period: int = Field(default=120, ge=1, description="ATR 週期")
    include_volatility_level: bool = Field(default=True, description="是否計算 ATR regime level")

    def create_strategy(self) -> StrategyConfig:
        builder = StrategyBuilder(params=self)

        daily_mp = builder.add_indicator(
            IndicatorType.MARKET_PROFILE,
            mode=create_daily_mode(self.ib_start_hour, 0),
            shape_config=create_profile_shape_config(
                pshape_concentration_threshold=0.5,
                bshape_valley_threshold=0.5,
            ),
            tick_size=self.tick_size,
            value_area_pct=0.70,
            freq=Freq.MIN_30,
            shift=0,
        )

        builder.add_indicator(
            IndicatorType.MARKET_PROFILE,
            mode=create_intraday_mode(self.ib_start_hour, 0, self.ib_end_hour, 0),
            shape_config=create_profile_shape_config(
                pshape_concentration_threshold=0.5,
                bshape_valley_threshold=0.5,
            ),
            tick_size=self.tick_size,
            value_area_pct=0.70,
            freq=Freq.MIN_30,
            shift=0,
        )

        atr = builder.add_indicator(
            IndicatorType.ATR,
            period=self.atr_period,
            freq=self.atr_freq,
            shift=1,
        )

        volatility_level = None
        if self.include_volatility_level:
            atr_daily = builder.add_indicator(IndicatorType.ATR, period=21, freq=Freq.DAY_1, shift=1)
            atr_daily_prev = builder.add_indicator(IndicatorType.ATR, period=21, freq=Freq.DAY_1, shift=2)
            atr_daily_q1 = builder.add_indicator(
                IndicatorType.ATR_QUANTILE,
                atr_column=atr_daily.display_name(),
                window=60,
                quantile=0.25,
                freq=Freq.DAY_1,
                shift=0,
            )
            atr_daily_q2 = builder.add_indicator(
                IndicatorType.ATR_QUANTILE,
                atr_column=atr_daily.display_name(),
                window=60,
                quantile=0.50,
                freq=Freq.DAY_1,
                shift=0,
            )
            atr_daily_q3 = builder.add_indicator(
                IndicatorType.ATR_QUANTILE,
                atr_column=atr_daily.display_name(),
                window=60,
                quantile=0.75,
                freq=Freq.DAY_1,
                shift=0,
            )
            volatility_level = VolatilityLevelConfig(
                source=atr_daily.col(),
                source_prev=atr_daily_prev.col(),
                q1=atr_daily_q1.col(),
                q2=atr_daily_q2.col(),
                q3=atr_daily_q3.col(),
            )

        vah = daily_mp.market_profile.vah.forward_fill()
        poc = daily_mp.market_profile.poc.forward_fill()
        entry_expr = pl.col("open") > vah
        exit_expr = pl.col("open") < poc

        base_bp = (
            BlueprintBuilder(
                name="mp_vah_poc",
                direction=self.trade_direction,
                trend_type=TrendType.REVERSAL,
            )
            .add_immediate_entry_trigger(conditions=[entry_expr])
            .add_immediate_exit_trigger(conditions=[exit_expr])
            .build()
        )

        builder.set_base_blueprint(base_bp)
        return builder.build(
            volatility_indicator=atr.col(),
            volatility_level=volatility_level,
            note="Minimal Market Profile example; entry open > VAH, exit open < POC.",
        )


## 建立 StrategyConfig

`StrategyConfig` 可以先理解成三個部分：

- `base_instrument` / `base_freq`：策略使用哪個商品與基準時間軸。
- `indicators`：需要計算的指標，本範例包含 Market Profile 與 ATR。indicator 內部也可以指定其他商品，例如策略交易 `PEPPERSTONE:spot:XAUUSD`，但某個 indicator 使用 `PEPPERSTONE:spot:NAS100`；後端會依 instrument / freq 自動載入並 join 到計算資料中。
- `base_blueprint`：極簡進出場規則，僅用來讓 `submit()` 能產生 trades。

`StrategyConfig` 的 instrument 請使用查詢結果 object 的 `inst.key`。例如可用 key 會長得像 `PEPPERSTONE:spot:XAUUSD` 或 `PEPPERSTONE:spot:NAS100`；選定後再把 key 放進策略參數。

`strategy.name` 是 SDK 用 zlib + base64url 壓縮出的 machine-readable reference。直接 `print(strategy.name)` 會看到 `sp:v1z:...`；用 `SimpleMarketProfileParams.decode_label()` 可以轉回可讀參數摘要。


In [8]:
# 推薦從上一格 instruments.instruments 的 object 取得 key。
# 例如：INSTRUMENT = instruments.instruments[0].key
INSTRUMENT = "PEPPERSTONE:spot:XAUUSD"

strategy = SimpleMarketProfileParams(
    instrument=INSTRUMENT,
    base_freq=Freq.MIN_15,
    trade_direction=TradeDirection.LONG,
    tick_size=1.0,
    atr_freq=Freq.HOUR_1,
    atr_period=120,
    include_volatility_level=True,
).create_strategy()

print(strategy.name)
print(SimpleMarketProfileParams.decode_label(strategy.name))
print(strategy.base_instrument, strategy.base_freq)
print(f"indicators={len(strategy.indicators)}")
print(f"base_blueprint={strategy.base_blueprint.name}")


sp:v1z:eNpljs1qw0AMhN9lz6HUhR66t5L6VloT29DbothyLbJ_0cqBJuTdo4SSS24zo4_RnAwIu4lxb6ypZrO6-YxMaTS2enlemS0UvBOvgaJCtHUYRzenhZWqbkERYPmP3jSJg19GdIfkQciT_DmPB_TGCi94vRcVAaNob1M3Tb1pu--v2pacxP689337oZ8iBFSgUMgeXQDeobjMaSK1GRhCUUpo2LlCR0WrJ90sDPp6JMZBKEUt-Ezx9woeVffdWuXjsLh4f74A9BJeJQ
{'name': 'simple_market_profile_params', 'atr_freq': '1h', 'atr_period': 120, 'base_freq': '15min', 'ib_end_hour': 11, 'ib_start_hour': 9, 'include_volatility_level': True, 'instrument': 'PEPPERSTONE:spot:XAUUSD', 'tick_size': 1.0, 'trade_direction': 'Long', 'tz': 'UTC'}
PEPPERSTONE:spot:XAUUSD Freq.MIN_15
indicators=8
base_blueprint=mp_vah_poc


In [9]:
# 簡短檢查 StrategyConfig JSON 結構。
strategy_json = strategy.model_dump(mode="json", exclude_none=True)
print(strategy_json.keys())
print("first indicator:")
strategy_json["indicators"][0]


dict_keys(['name', 'base_instrument', 'base_freq', 'note', 'tz', 'volatility_indicator', 'indicators', 'data_sources', 'volatility_level', 'base_blueprint', 'advanced_blueprints'])
first indicator:


{'instrument': 'PEPPERSTONE:spot:XAUUSD',
 'freq': '30min',
 'shift': 0,
 'indicator': {'type': 'MarketProfile',
  'mode': {'mode': 'Daily', 'hour': 9, 'minute': 0},
  'tick_size': 1.0,
  'value_area_pct': 0.7,
  'shape_config': {'early_period_ratio': 0.15,
   'late_period_ratio': 0.15,
   'trend_ib_max_ratio': 0.2,
   'trend_monotonic_threshold': 0.6,
   'trend_imbalance_threshold': 0.7,
   'pshape_concentration_threshold': 0.5,
   'bshape_valley_threshold': 0.5,
   'normal_symmetry_threshold': 0.3}}}

## 取回 Market Profile 指標計算結果

`submit_ohlcv()` 會從 strategy 抽出所有 indicator，提交 on-demand OHLCV task，背景輪詢下載完成後可從 `ohlcv_result.data` 或 `ohlcv_result.df` 讀到基礎 OHLCV 加上指標欄位。

注意：`OHLCVPeriodResult` 不提供 `wait()` method，所以這裡提交後先短暫等待，再讀 `.data`。如果資料尚未下載完成，可以稍後重新讀 `ohlcv_result.data`。


In [10]:
period = Period(start="2025-01-01", end="2027-02-01")

ohlcv_result = tester.submit_ohlcv(strategy=strategy, period=period, timeout=300)

import time
time.sleep(5)

ohlcv_df = ohlcv_result.data
ohlcv_df

ts,open,high,low,close,volume,PEPPERSTONE:spot:XAUUSD_30min_MP|D_0900_1_s0,PEPPERSTONE:spot:XAUUSD_30min_MP|IB_0900_1100_1_s0,PEPPERSTONE:spot:XAUUSD_1h_ATR|120,PEPPERSTONE:spot:XAUUSD_1D_ATR|21,PEPPERSTONE:spot:XAUUSD_1D_ATR|21_s2,PEPPERSTONE:spot:XAUUSD_1D_ATRQ|PEPPERSTONE:spot:XAUUSD_1D_ATR|21_Q25_60_s0,PEPPERSTONE:spot:XAUUSD_1D_ATRQ|PEPPERSTONE:spot:XAUUSD_1D_ATR|21_Q50_60_s0,PEPPERSTONE:spot:XAUUSD_1D_ATRQ|PEPPERSTONE:spot:XAUUSD_1D_ATR|21_Q75_60_s0
datetime[ms],f64,f64,f64,f64,f64,struct[7],struct[7],f64,f64,f64,f64,f64,f64
2025-01-01 23:00:00,2624.91,2625.71,2624.06,2624.6,533.0,"{null,null,null,null,null,null,null}","{null,null,null,null,null,null,null}",null,null,null,null,null,null
2025-01-01 23:15:00,2624.64,2625.03,2623.27,2623.44,492.0,"{null,null,null,null,null,null,null}","{null,null,null,null,null,null,null}",null,null,null,null,null,null
2025-01-01 23:30:00,2623.44,2623.95,2621.75,2621.75,470.0,"{null,null,null,null,null,null,null}","{null,null,null,null,null,null,null}",null,null,null,null,null,null
2025-01-01 23:45:00,2621.67,2624.5,2621.58,2623.76,530.0,"{null,null,null,null,null,null,null}","{null,null,null,null,null,null,null}",null,null,null,null,null,null
2025-01-02 00:00:00,2623.95,2626.09,2622.58,2625.76,673.0,"{null,null,null,null,null,null,null}","{null,null,null,null,null,null,null}",null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-04-27 13:45:00,4693.31,4697.41,4689.16,4696.11,3980.0,"{null,null,null,null,null,null,null}","{null,null,null,null,null,null,null}",20.071988,122.204731,126.658467,140.202592,147.852524,157.035714
2026-04-27 14:00:00,4695.77,4696.34,4687.82,4692.41,3897.0,"{null,null,null,null,null,null,null}","{null,null,null,null,null,null,null}",20.022138,122.204731,126.658467,140.202592,147.852524,157.035714
2026-04-27 14:15:00,4692.36,4696.71,4686.09,4689.07,3806.0,"{null,null,null,null,null,null,null}","{null,null,null,null,null,null,null}",20.022138,122.204731,126.658467,140.202592,147.852524,157.035714


In [11]:
# Market Profile 指標欄位是 struct；display_name 會跟 instrument / freq / mode / tick_size 變動。
market_profile_cols = [
    spec.display_name()
    for spec in strategy.indicators
    if spec.indicator.type == "MarketProfile"
]
daily_mp_col, ib_mp_col = market_profile_cols

mb = pl.col(daily_mp_col).struct.field("tpo_distribution")
ib = pl.col(ib_mp_col).struct.field("tpo_distribution")

df = ohlcv_result.df.filter(mb.is_not_null() | ib.is_not_null())

row = df.head(1).to_dicts()[0]
mp_data = row[daily_mp_col].pop("tpo_distribution")[0]

row

{'ts': datetime.datetime(2025, 1, 2, 9, 0),
 'open': 2636.45,
 'high': 2639.87,
 'low': 2636.45,
 'close': 2638.28,
 'volume': 1659.0,
 'PEPPERSTONE:spot:XAUUSD_30min_MP|D_0900_1_s0': {'segment_id': 1,
  'poc': 2633.0,
  'vah': 2637.0,
  'val': 2631.0,
  'value_area': 6.0,
  'profile_shape': 'p_shaped'},
 'PEPPERSTONE:spot:XAUUSD_30min_MP|IB_0900_1100_1_s0': {'segment_id': None,
  'poc': None,
  'vah': None,
  'val': None,
  'value_area': None,
  'tpo_distribution': None,
  'profile_shape': None},
 'PEPPERSTONE:spot:XAUUSD_1h_ATR|120': None,
 'PEPPERSTONE:spot:XAUUSD_1D_ATR|21': None,
 'PEPPERSTONE:spot:XAUUSD_1D_ATR|21_s2': None,
 'PEPPERSTONE:spot:XAUUSD_1D_ATRQ|PEPPERSTONE:spot:XAUUSD_1D_ATR|21_Q25_60_s0': None,
 'PEPPERSTONE:spot:XAUUSD_1D_ATRQ|PEPPERSTONE:spot:XAUUSD_1D_ATR|21_Q50_60_s0': None,
 'PEPPERSTONE:spot:XAUUSD_1D_ATRQ|PEPPERSTONE:spot:XAUUSD_1D_ATR|21_Q75_60_s0': None}

In [12]:
pl.Config.set_fmt_str_lengths(10_000)
pl.Config.set_fmt_table_cell_list_len(100)
pl.Config.set_tbl_rows(100)

mp_df = pl.DataFrame(mp_data)
mp_df.sort("price")[::-1]

price,count,periods
f64,i64,list[i64]
2638.0,1,[19]
2637.0,4,"[6, 13, 18, 19]"
2636.0,7,"[6, 7, 13, 16, 17, 18, 19]"
2635.0,10,"[6, 7, 8, 12, 13, 15, 16, 17, 18, 19]"
2634.0,15,"[5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]"
2633.0,15,"[4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16, 17, 18, 19]"
2632.0,9,"[4, 5, 7, 9, 10, 11, 14, 18, 19]"
2631.0,5,"[4, 11, 14, 18, 19]"
2630.0,1,[4]


## 用 submit() 產生 Trades

`tester.submit()` 會依照極簡進出場規則產生 trades。這裡只帶到流程；產生後的 trades 會包含 MAE、MFE、PnL 等欄位，可供後續分析、篩選、分組與視覺化使用。


In [13]:
batch = tester.submit(strategies=[strategy], periods=[period])
print(batch.status)

batch.wait(timeout=600)

summary = batch.summary()
trades = batch.all_trades()

print(summary)
print(trades.shape)
trades.tail(20)

2026-04-27 22:54:34,517 - tradepose_client.batch.tester - INFO - Submitted 1 batch tasks (1 strategies × 1 periods)


{'pending': 1, 'processing': 0, 'completed': 0, 'failed': 0}
shape: (2, 21)
┌────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ trades ┆ first_entr ┆ last_entry ┆ last_exit ┆ … ┆ metric_ty ┆ strategy_ ┆ blueprint ┆ period    │
│ ---    ┆ y_time     ┆ _time      ┆ _time     ┆   ┆ pe        ┆ name      ┆ _name     ┆ ---       │
│ u32    ┆ ---        ┆ ---        ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ str       │
│        ┆ datetime[m ┆ datetime[m ┆ datetime[ ┆   ┆ str       ┆ str       ┆ str       ┆           │
│        ┆ s]         ┆ s]         ┆ ms]       ┆   ┆           ┆           ┆           ┆           │
╞════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 230    ┆ 2025-01-02 ┆ 2026-04-24 ┆ 2026-04-2 ┆ … ┆ pnl       ┆ sp:v1z:eN ┆ mp_vah_po ┆ 2025-01-0 │
│        ┆ 09:15:00   ┆ 14:15:00   ┆ 6         ┆   ┆           ┆ pljs1qw0A ┆ c         ┆ 1_2027-02 │
│        ┆     

direction,status,is_adv_triggered,entry_idx,entry_time,entry_price,entry_reason,favorable_entry_hypo_price,adverse_entry_hypo_price,favorable_entry_strategy,adverse_entry_strategy,neutral_entry_strategy,exit_idx,exit_time,exit_price,exit_reason,favorable_exit_hypo_price,adverse_exit_hypo_price,favorable_exit_strategy,adverse_exit_strategy,neutral_exit_strategy,holding_seconds,holding_bars,g_mfe,mae,mfe,mae_lv1,mhl,g_mfe_idx,mae_idx,mfe_idx,mae_lv1_idx,mhl_idx,entry_volatility,exit_volatility,g_mfe_volatility,mae_volatility,mfe_volatility,mae_lv1_volatility,mhl_volatility,entry_volatility_level,base_entry_time,base_entry_price,pnl,pnl_pct,strategy_name,blueprint_name,period
i16,bool,bool,u32,datetime[ms],f64,i16,f64,f64,i16,i16,i16,u32,datetime[ms],f64,i16,f64,f64,i16,i16,i16,i64,i32,f64,f64,f64,f64,f64,u32,u32,u32,u32,u32,f64,f64,f64,f64,f64,f64,f64,u8,datetime[ms],f64,f64,f64,str,str,str
1,false,false,27354,2026-02-27 13:15:00,5212.84,0,null,null,null,null,0,27451,2026-03-02 14:30:00,5333.12,3,null,null,null,null,3,263700,98,206.52,5.2,27.52,0.0,112.69,27422,27355,27354,null,27450,24.795408,25.945214,25.965483,24.795408,24.795408,null,25.945214,3,2026-02-27 13:15:00,5212.84,120.28,0.023074,"""sp:v1z:eNpljs1qw0AMhN9lz6HUhR66t5L6VloT29DbothyLbJ_0cqBJuTdo4SSS24zo4_RnAwIu4lxb6ypZrO6-YxMaTS2enlemS0UvBOvgaJCtHUYRzenhZWqbkERYPmP3jSJg19GdIfkQciT_DmPB_TGCi94vRcVAaNob1M3Tb1pu--v2pacxP689337oZ8iBFSgUMgeXQDeobjMaSK1GRhCUUpo2LlCR0WrJ90sDPp6JMZBKEUt-Ezx9woeVffdWuXjsLh4f74A9BJeJQ""","""mp_vah_poc""","""2025-01-01_2027-02-01"""
1,false,false,27618,2026-03-04 10:15:00,5198.34,0,null,null,null,null,0,27637,2026-03-04 15:00:00,5146.75,3,null,null,null,null,3,17100,20,7.78,59.97,7.78,18.9,67.75,27625,27636,27625,27619,27636,32.543589,32.543651,32.306764,32.424691,32.306764,32.543589,32.424691,3,2026-03-04 10:15:00,5198.34,-51.59,-0.009924,"""sp:v1z:eNpljs1qw0AMhN9lz6HUhR66t5L6VloT29DbothyLbJ_0cqBJuTdo4SSS24zo4_RnAwIu4lxb6ypZrO6-YxMaTS2enlemS0UvBOvgaJCtHUYRzenhZWqbkERYPmP3jSJg19GdIfkQciT_DmPB_TGCi94vRcVAaNob1M3Tb1pu--v2pacxP689337oZ8iBFSgUMgeXQDeobjMaSK1GRhCUUpo2LlCR0WrJ90sDPp6JMZBKEUt-Ezx9woeVffdWuXjsLh4f74A9BJeJQ""","""mp_vah_poc""","""2025-01-01_2027-02-01"""
1,false,false,27681,2026-03-05 03:00:00,5192.08,0,null,null,null,null,0,27693,2026-03-05 06:00:00,5148.62,3,null,null,null,null,3,10800,13,2.85,43.95,2.85,0.0,46.8,27681,27692,27681,null,27692,31.81823,31.57415,31.81823,31.598891,31.81823,null,31.598891,3,2026-03-05 03:00:00,5192.08,-43.46,-0.00837,"""sp:v1z:eNpljs1qw0AMhN9lz6HUhR66t5L6VloT29DbothyLbJ_0cqBJuTdo4SSS24zo4_RnAwIu4lxb6ypZrO6-YxMaTS2enlemS0UvBOvgaJCtHUYRzenhZWqbkERYPmP3jSJg19GdIfkQciT_DmPB_TGCi94vRcVAaNob1M3Tb1pu--v2pacxP689337oZ8iBFSgUMgeXQDeobjMaSK1GRhCUUpo2LlCR0WrJ90sDPp6JMZBKEUt-Ezx9woeVffdWuXjsLh4f74A9BJeJQ""","""mp_vah_poc""","""2025-01-01_2027-02-01"""
1,false,false,27821,2026-03-06 15:00:00,5130.4,0,null,null,null,null,0,27856,2026-03-09 00:45:00,5072.83,3,null,null,null,null,3,207900,36,61.68,86.12,61.68,4.58,147.8,27849,27855,27849,27827,27855,31.37942,31.494105,31.019182,31.494105,31.019182,31.481425,31.494105,3,2026-03-06 15:00:00,5130.4,-57.57,-0.011221,"""sp:v1z:eNpljs1qw0AMhN9lz6HUhR66t5L6VloT29DbothyLbJ_0cqBJuTdo4SSS24zo4_RnAwIu4lxb6ypZrO6-YxMaTS2enlemS0UvBOvgaJCtHUYRzenhZWqbkERYPmP3jSJg19GdIfkQciT_DmPB_TGCi94vRcVAaNob1M3Tb1pu--v2pacxP689337oZ8iBFSgUMgeXQDeobjMaSK1GRhCUUpo2LlCR0WrJ90sDPp6JMZBKEUt-Ezx9woeVffdWuXjsLh4f74A9BJeJQ""","""mp_vah_poc""","""2025-01-01_2027-02-01"""
1,false,false,27935,2026-03-09 20:30:00,5140.02,0,null,null,null,null,0,28075,2026-03-11 09:30:00,5184.7,3,null,null,null,null,3,133200,141,98.59,22.53,10.07,14.85,77.88,28012,27942,27941,27935,28022,32.200292,29.754771,30.948453,31.994637,31.994637,32.200292,30.679197,3,2026-03-09 20:30:00,5140.02,44.68,0.008693,"""sp:v1z:eNpljs1qw0AMhN9lz6HUhR66t5L6VloT29DbothyLbJ_0cqBJuTdo4SSS24zo4_RnAwIu4lxb6ypZrO6-YxMaTS2enlemS0UvBOvgaJCtHUYRzenhZWqbkERYPmP3jSJg19GdIfkQciT_DmPB_TGCi94vRcVAaNob1M3Tb1pu--v2pacxP689337oZ8iBFSgUMgeXQDeobjMaSK1GRhCUUpo2LlCR0W

In [ ]:
# 可選：保留結果供後續分析。Colab 會寫到 session 檔案系統。
ohlcv_df.write_parquet("simple_market_profile_ohlcv.parquet")
trades.write_parquet("simple_market_profile_trades.parquet")
print("saved parquet files")
